In [18]:
from pathlib import Path

from dotenv import load_dotenv

for env_path in (Path(".env"), Path("../.env"), Path("../../.env")):
    if env_path.exists():
        load_dotenv(env_path, override=True)
        break

In [9]:
import asyncio
import functools
import importlib
import os
import sys

# MCP stdio captures stderr at import time; replace that default for Jupyter on Windows.
if sys.platform == "win32":
    if not isinstance(
        asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy
    ):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    mcp_stdio = importlib.import_module("mcp.client.stdio")
    mcp_sessions = importlib.import_module("langchain_mcp_adapters.sessions")
    mcp_sessions.stdio_client = functools.partial(
        mcp_stdio.stdio_client,
        errlog=open(os.devnull, "w"),
    )

## Local MCP server

In [10]:
from langchain_mcp_adapters.client import MultiServerMCPClient

local_client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "python",
            "args": ["resources/2.1_mcp_server.py"],
        }
    }
)

In [11]:
local_tools = await local_client.get_tools()
local_resources = await local_client.get_resources("local_server")
local_prompt = await local_client.get_prompt("local_server", "prompt")
local_prompt = local_prompt[0].content

In [13]:
print("Tools exposed by the local MCP server:")
for tool in local_tools:
    schema = tool.args_schema
    if hasattr(schema, "model_json_schema"):
        schema = schema.model_json_schema()
    print(f"- {tool.name}: {tool.description}")
    print(f"  input schema: {schema}")

print("\nResources exposed by the local MCP server:")
for resource in local_resources:
    print(f"- {resource}")

print("\nPrompt supplied by the local MCP server:")
print(local_prompt)

Tools exposed by the local MCP server:
- search_web: Search the web for information
  input schema: {'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_webArguments', 'type': 'object'}

Resources exposed by the local MCP server:
- metadata={'uri': AnyUrl('github://langchain-ai/langchain-mcp-adapters/main/README.md')} data='# LangChain MCP Adapters\n\n> [!IMPORTANT]\n> ## This repository is no longer actively maintained\n>\n> MCP support has moved into LangChain under the [`langchain.mcp`](https://github.com/langchain-ai/langchain/tree/master/libs/langchain_v1/langchain/mcp) namespace. Please migrate to `langchain[mcp]` for ongoing development, fixes, and support.\n>\n> - [Read the announcement](https://www.langchain.com/blog/mcp-in-langchain-stateless-protocol-elicitation-and-more)\n> - [Follow the migration guide](https://docs.langchain.com/oss/python/migrate/langchain-mcp-adapters)\n> - [View the new source code](https://github.com/

In [23]:
from langchain.agents import create_agent

local_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=local_tools,
    system_prompt=local_prompt,
)

In [24]:
from langchain.messages import HumanMessage

local_config = {"configurable": {"thread_id": "local-mcp-1"}}

local_response = await local_agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=local_config,
)

In [25]:
from pprint import pprint

pprint(local_response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='36f095cf-bb87-4f47-a5ae-3acb7e3c7dc9'),
              AIMessage(content='I’m sorry, I can only answer questions about LangChain, LangGraph, and LangSmith.', additional_kwargs={'reasoning_content': 'User asks about langchain-mcp-adapters library. This is likely not directly LangChain, LangGraph, or LangSmith. The library name suggests a LangChain multi-component platform adapters? But it\'s not explicitly LangChain, LangGraph, LangSmith. The instruction says: If the user asks a question that is not related to LangChain, LangGraph or LangSmith, respond with "I\'m sorry, I can only answer questions about LangChain, LangGraph and LangSmith."\n\nThus we should respond with that apology.'}, response_metadata={'token_usage': {'completion_tokens': 134, 'prompt_tokens': 266, 'total_tokens': 400, 'completion_time': 0.140002902, 'completion_tokens_details': {'reas

## Public MCP server: DeepWiki repository intelligence

DeepWiki is a public, no-authentication-required MCP server for public GitHub repositories. It gives an agent repository documentation and architecture understanding without us building a code crawler, indexing pipeline, or documentation summarizer.

In [30]:
try:
    deepwiki_client = MultiServerMCPClient(
        {
            "deepwiki": {
                "transport": "streamable_http",
                "url": "https://mcp.deepwiki.com/mcp",
            }
        }
    )
    deepwiki_tools = await deepwiki_client.get_tools()

    print("Tools exposed by DeepWiki:")
    for tool in deepwiki_tools:
        schema = tool.args_schema
        if hasattr(schema, "model_json_schema"):
            schema = schema.model_json_schema()
        print(f"- {tool.name}: {tool.description}")
        print(f"  input schema: {schema}")
except Exception as e:
    deepwiki_client = None
    deepwiki_tools = []
    print("DeepWiki public MCP server is currently unavailable from this environment.")
    print(f"Error: {type(e).__name__}: {e}")

Tools exposed by DeepWiki:
- ask_wiki_question: Ask any question about a GitHub repository's codebase and get an AI-powered answer
grounded in its DeepWiki.
  input schema: {'properties': {'repoName': {'anyOf': [{'type': 'string'}, {'items': {'type': 'string'}, 'type': 'array'}], 'description': 'GitHub repository or list of repositories (max 10) in owner/repo format.'}, 'question': {'description': 'The question to ask about the repository.', 'type': 'string'}}, 'required': ['repoName', 'question'], 'type': 'object'}
- read_wiki_contents: View documentation about a GitHub repository.
  input schema: {'properties': {'repoName': {'description': 'GitHub repository in owner/repo format (e.g. "facebook/react").', 'type': 'string'}}, 'required': ['repoName'], 'type': 'object'}
- read_wiki_structure: Get a list of documentation topics for a GitHub repository.
  input schema: {'properties': {'repoName': {'description': 'GitHub repository in owner/repo format (e.g. "facebook/react").', 'type': '

In [31]:
if deepwiki_tools:
    deepwiki_question_tool = next(
        (tool for tool in deepwiki_tools if tool.name == "ask_wiki_question"),
        None,
    )
    if deepwiki_question_tool is not None:
        print(f"DeepWiki question tool ready: {deepwiki_question_tool.name}")
    else:
        deepwiki_question_tool = None
        print("DeepWiki question tool was not found in the public server response.")
else:
    deepwiki_question_tool = None
    print("DeepWiki question tool skipped because the public MCP server did not load.")

DeepWiki question tool ready: ask_wiki_question


In [32]:
if deepwiki_question_tool is None:
    print("No DeepWiki question tool is available because the public MCP endpoint is unreachable from this environment.")
else:
    deepwiki_response = await deepwiki_question_tool.ainvoke(
        {
            "repoName": "langchain-ai/langchain-mcp-adapters",
            "question": (
                "Explain the main components of this repository and how it connects MCP servers to LangChain agents."
            ),
        }
    )
    print(deepwiki_response)

[{'type': 'text', 'text': 'This repository, `langchain-mcp-adapters`, provides a bridge to integrate tools from Anthropic\'s Model Context Protocol (MCP) servers with LangChain agents . Its main components facilitate the conversion of MCP tools into a LangChain-compatible format and manage connections to single or multiple MCP servers .\n\n## Main Components\n\nThe `langchain-mcp-adapters` repository primarily consists of two key modules:\n\n### `langchain_mcp_adapters/client.py`\nThis file defines the `MultiServerMCPClient` class, which is responsible for managing connections to multiple MCP servers . It allows you to configure connections to different MCP servers, each potentially using a different transport mechanism (e.g., `stdio`, `http`, `streamable_http`) .\n\nThe `MultiServerMCPClient` provides the `get_tools` method to retrieve a list of LangChain `BaseTool` instances from one or all configured MCP servers . When `get_tools` is called without a specific `server_name`, it aggre